In [ ]:
import mlflow

mlflow.set_experiment("Mushroom Categorization")
mlflow.autolog()

In [ ]:
import pandas as pd

df = pd.read_parquet('s3://traindata/train_raw.parquet', storage_options={"anon": False})

categoricals = ['cap-shape', 'gill-attachment', 'gill-color', 'stem-color']
numericals = [c for c in df.drop('class', axis='columns').columns if c not in categoricals]

for col in df.columns:
    df[col] = df[col].astype('float')

In [ ]:
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from mlflow.models import infer_signature

X_train, X_test, y_train, y_test = train_test_split(df.drop('class', axis='columns'), df['class'], random_state=42)
X_train_small, X_val, y_train_small, y_val = train_test_split(X_train, y_train, random_state=42)


@mlflow.trace
def train(X: pd.DataFrame, y: pd.DataFrame, X_validation: pd.DataFrame, y_validation: pd.DataFrame, n_estimators=10,
          n_folds=10):
    model_name = f"baseline-{n_estimators ** 1 + n_folds ** 2}"
    with mlflow.start_run(run_name=f"{model_name}@{datetime.today().strftime('%Y-%m-%d_%H%M%S')}") as run:
        mlflow.sklearn.autolog(registered_model_name=model_name, log_models=False)
        print(f"\n# Model {model_name}")

        params = {
            'n_estimators': n_estimators
        }
        mlflow.log_params(params)
        clf = GradientBoostingClassifier(**params)

        scores = cross_val_score(clf, X, y, cv=n_folds)
        print(f"\n## Cross Validation")
        print(
            f"Our classifier has an accuracy of {scores.mean():0.2f} (standard deviation {scores.std():0.2f}) over all {n_folds} folds")

        clf.fit(X, y)
        # or without autolog
        # signature = infer_signature(model_input=X, model_output=y)
        # mlflow.sklearn.log_model(
        #     sk_model=clf, signature=signature, name=model_name,
        #     registered_model_name=model_name,
        #     serialization_format="skops",
        #     skops_trusted_types=['sklearn.tree._tree.Tree'],
        # )

        y_proba = clf.predict(X_validation)

        metrics_static_val_set = {
            "accuracy": accuracy_score(y_validation, y_proba),
            "precision": precision_score(y_validation, y_proba),
            "recall": recall_score(y_validation, y_proba),
            "f1": f1_score(y_validation, y_proba),
        }
        # or without autolog
        # mlflow.log_metrics(metrics_static_val_set)

        print(f"\n## Metrics")
        print(f"{"\t\t".join([f"{x}: {metrics_static_val_set[x]}" for x in metrics_static_val_set.keys()])}")

In [ ]:
train(X_train_small, y_train_small, X_val, y_val, 10, 10)
train(X_train_small, y_train_small, X_val, y_val, 20, 20)
train(X_train_small, y_train_small, X_val, y_val, 20, 10)
train(X_train_small, y_train_small, X_val, y_val, 30, 10)
train(X_train_small, y_train_small, X_val, y_val, 30, 20)
train(X_train_small, y_train_small, X_val, y_val, 30, 30)